# Import data from Tricoteuses

Useful Links :
- Site des Tricoteuses : https://www.tricoteuses.fr, https://parlement.tricoteuses.fr/docs#description/introduction
- une API développée pour récupérer les données Open de l'assemblée et du Sénat : https://git.tricoteuses.fr/logiciels/tricoteuses-api-parlement
- Le code de récupération et nettoyage des données Open : https://git.tricoteuses.fr/logiciels/tricoteuses-assemblee/src/branch/master

rq : how to get the max of rows without overestimating the perPage param

next_step : 
- filter on circo = null to exclude minister
- Test again API for Collaborateurs and Adresse

In [3]:
import requests
import pandas as pd
import json

response_tr = requests.get(
    "https://parlement.tricoteuses.fr/v2/acteurs/",
        params={
          "searchScope": "name",
            "chambre" : "AN",
            "page" : "1",
            "actif" : "true",
            "perPage" : "1000"
        }
)

data_tr = response_tr.json()

df_tr = pd.json_normalize(data_tr['data'])
print(df_tr)

          uid         nom          prenom  civ chambre  actif  \
0      PA1008       David           Alain   M.      AN   True   
1      PA1327   Forissier         Nicolas   M.      AN   True   
2      PA1567       Guedj          Jérôme   M.      AN   True   
3      PA1592       Habib           David   M.      AN   True   
4      PA1630   Herbillon          Michel   M.      AN   True   
..        ...         ...             ...  ...     ...    ...   
591  PA870009     Duparay          Lionel   M.      AN   True   
592  PA870010      Martin       Sébastien   M.      AN   True   
593  PA878242    Valentin         Antoine   M.      AN   True   
594  PA879389       Lubet  Marie-Philippe  Mme      AN   True   
595     PA942  de Courson         Charles   M.      AN   True   

                                              urlImage  \
0    https://tricoteuses-assets.s3.fr-par.scw.cloud...   
1    https://tricoteuses-assets.s3.fr-par.scw.cloud...   
2    https://tricoteuses-assets.s3.fr-par.scw

In [4]:
df_tr.info()

<class 'pandas.DataFrame'>
RangeIndex: 596 entries, 0 to 595
Data columns (total 33 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   uid                                  596 non-null    str    
 1   nom                                  596 non-null    str    
 2   prenom                               596 non-null    str    
 3   civ                                  596 non-null    str    
 4   chambre                              596 non-null    str    
 5   actif                                596 non-null    bool   
 6   urlImage                             596 non-null    str    
 7   interets                             596 non-null    object 
 8   placeHemicycle                       574 non-null    str    
 9   compteTwitter                        370 non-null    str    
 10  compteBluesky                        180 non-null    str    
 11  groupeParlementaire.uid              574 no

In [5]:
uid_tr = df_tr[['uid']]
uid_tr['source'] = 'tricoteuse' 
print(uid_tr)

          uid      source
0      PA1008  tricoteuse
1      PA1327  tricoteuse
2      PA1567  tricoteuse
3      PA1592  tricoteuse
4      PA1630  tricoteuse
..        ...         ...
591  PA870009  tricoteuse
592  PA870010  tricoteuse
593  PA878242  tricoteuse
594  PA879389  tricoteuse
595     PA942  tricoteuse

[596 rows x 2 columns]


## Tests avec d'autres tables récupérables via l'API

### Collaborateurs

In [6]:
response_co = requests.get(
    "https://parlement.tricoteuses.fr/collaborateurs/json",
    params={
      "page": "1",
      "perPage": "10",
      "distinct": [
        "uid",
        "datePremierAjout",
        "dateMaj"
      ],
      "searchLanguage": "french",
      "search": "loi travail",
      "sort": "nom.asc"
    }
)
print(response_co)

#data_co = response_co.json()

#df_co = pd.json_normalize(data_co['data'])
#print(df_co)

<Response [500]>


In [7]:
requests.get(
    "https://parlement.tricoteuses.fr/adressesElectroniques/json",
    params={
      "page": "1",
      "perPage": "10",
      "distinct": [
        "uid",
        "datePremierAjout",
        "dateMaj"
      ],
    }
)

<Response [500]>

# Import data from datan database

Useful link:
- feuille de l'explorateur de données de la base des Député Actif : [LIEN](https://explore.data.gouv.fr/fr/datasets/5fc8b732d30fbf1ed6648aab/#/resources/092bd7bb-1543-405b-b53c-932ebb49bb8e)
- https://www.data.gouv.fr/datasets/deputes-actifs-de-lassemblee-nationale-informations-et-statistiques
- https://www.data.gouv.fr/dataservices/api-tabulaire-data-gouv-fr-beta

In [8]:
#rid of the dataset imported from https://explore.data.gouv.fr/fr/datasets/5fc8b732d30fbf1ed6648aab/#/resources/092bd7bb-1543-405b-b53c-932ebb49bb8e
rid = "092bd7bb-1543-405b-b53c-932ebb49bb8e"

response_dn = requests.get(
    "https://tabular-api.data.gouv.fr/api/resources/" + rid + "/data/json/",
)

data_dn = response_dn.json()

df_dn = pd.json_normalize(data_dn)

print(df_dn)

     __id        id  legislature  civ         nom          prenom  \
0       1    PA1008           17   M.       David           Alain   
1       2    PA1567           17   M.       Guedj          Jérôme   
2       3    PA1592           17   M.       Habib           David   
3       4    PA1630           17   M.   Herbillon          Michel   
4       5    PA1654           17   M.    Hollande        François   
..    ...       ...          ...  ...         ...             ...   
569   570  PA857021           17   M.    Latrèche            Alim   
570   571  PA870009           17   M.     Duparay          Lionel   
571   572  PA878242           17   M.    Valentin         Antoine   
572   573  PA879389           17  Mme       Lubet  Marie-Philippe   
573   574     PA942           17   M.  de Courson         Charles   

       villeNaissance   naissance  age  \
0            Libourne  1949-06-02   77   
1              Pantin  1972-01-23   54   
2               Paris  1961-03-16   65   
3  

In [9]:
df_dn.info()

<class 'pandas.DataFrame'>
RangeIndex: 574 entries, 0 to 573
Data columns (total 27 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   __id                          574 non-null    int64  
 1   id                            574 non-null    str    
 2   legislature                   574 non-null    int64  
 3   civ                           574 non-null    str    
 4   nom                           574 non-null    str    
 5   prenom                        574 non-null    str    
 6   villeNaissance                574 non-null    str    
 7   naissance                     574 non-null    str    
 8   age                           574 non-null    int64  
 9   groupe                        574 non-null    str    
 10  groupeAbrev                   574 non-null    str    
 11  departementNom                574 non-null    str    
 12  departementCode               574 non-null    str    
 13  circo           

In [10]:
#get Deputy Ids and rename column
uid_dn = df_dn[["id"]]
uid_dn = uid_dn.rename(columns={"id":"uid"})
#add source column to sub dataframe
uid_dn['source']='datan'
print(uid_dn)

          uid source
0      PA1008  datan
1      PA1567  datan
2      PA1592  datan
3      PA1630  datan
4      PA1654  datan
..        ...    ...
569  PA857021  datan
570  PA870009  datan
571  PA878242  datan
572  PA879389  datan
573     PA942  datan

[574 rows x 2 columns]


# Comparison of the sub data frames : what are the ids that differ?

In [11]:
#concat the 2 sub dataframes
uid = pd.concat([uid_tr,uid_dn])
print(uid)

          uid      source
0      PA1008  tricoteuse
1      PA1327  tricoteuse
2      PA1567  tricoteuse
3      PA1592  tricoteuse
4      PA1630  tricoteuse
..        ...         ...
569  PA857021       datan
570  PA870009       datan
571  PA878242       datan
572  PA879389       datan
573     PA942       datan

[1170 rows x 2 columns]


In [12]:
#get the ids
df_filter = uid.drop_duplicates(subset=["uid"], keep=False)
print(df_filter)

          uid      source
1      PA1327  tricoteuse
17   PA267797  tricoteuse
41   PA408578  tricoteuse
46   PA605991  tricoteuse
55   PA607846  tricoteuse
109  PA719736  tricoteuse
122  PA720066  tricoteuse
151  PA720908  tricoteuse
160  PA721134  tricoteuse
181  PA721836  tricoteuse
191  PA722046  tricoteuse
226  PA793210  tricoteuse
234  PA793278  tricoteuse
310  PA794426  tricoteuse
361  PA795120  tricoteuse
377  PA795330  tricoteuse
382  PA795386  tricoteuse
419  PA795950  tricoteuse
421  PA795958  tricoteuse
426  PA795990  tricoteuse
573  PA842117  tricoteuse
592  PA870010  tricoteuse


In [13]:
#inner join to filter the dataframe from Tricoteuses
result_diff = df_tr.join(df_filter.set_index("uid"),on="uid",how='inner')
print(result_diff)

          uid              nom       prenom  civ chambre  actif  \
1      PA1327        Forissier      Nicolas   M.      AN   True   
17   PA267797          Vautrin    Catherine  Mme      AN   True   
41   PA408578           Berger  Jean-Didier   M.      AN   True   
46   PA605991         Genevard        Annie  Mme      AN   True   
55   PA607846         Darmanin       Gérald   M.      AN   True   
109  PA719736  Galliard-Minier      Camille  Mme      AN   True   
122  PA720066             Rist    Stéphanie  Mme      AN   True   
151  PA720908         Moutchou        Naïma  Mme      AN   True   
160  PA721134          Lescure       Roland   M.      AN   True   
181  PA721836           Barrot    Jean-Noël   M.      AN   True   
191  PA722046            Bergé       Aurore  Mme      AN   True   
226  PA793210         Panifous      Laurent   M.      AN   True   
234  PA793278         Roubache      Sabrina  Mme      AN   True   
310  PA794426       Le Hénanff         Anne  Mme      AN   Tru

Les noms listés en plus dans la base des Tricoteuses semblent appartenir à des députés passés ministres :
- Nicolas Forissier : Ministre délégué auprès du ministre de l’Europe et des Affaires étrangères, chargé du Commerce extérieur et de l'Attractivité
- Catherine Vautrin : Ministre des Armées et des Anciens combattants
- Jean-Didier Berger : Ministre délégué auprès du ministre de l'Intérieur
- Annie Genevard : Ministre de l’Agriculture, de l’Agro-alimentaire et de la Souveraineté alimentaire
- Gérald Darmanin : Garde des Sceaux, ministre de la Justice
- Camille Galliard-Minier : Ministre déléguée, auprès de la ministre de la Santé, des Familles, de l’Autonomie et des Personnes handicapées, chargée de l’Autonomie et des Personnes handicapées
- Stéphanie Rist : Ministre de la Santé, des Familles, de l’Autonomie et des Personnes handicapées
- Naïma Moutchou : Ministre des Outre-mer
- Roland Lescure : Ministre de l’Économie, des Finances et de la Souveraineté industrielle, énergétique et numérique
- Jean-Noël Barrot : Ministre de l’Europe et des Affaires étrangères
- Aurore Bergé : Ministre déléguée auprès du Premier ministre, chargée de l'Égalité entre les femmes et les hommes et de la Lutte contre les discriminations
- Laurent Panifous : Ministre délégué auprès du Premier ministre, chargé des Relations avec le Parlement
- Sabrina Roubache : Ministre déléguée auprès du ministre du Travail et des Solidarités et du ministre de l'Éducation nationale, chargée de l'Enseignement, et de la Formation professionnels et de l’Apprentissage
- Anne Le Hénanff : Ministre déléguée auprès du ministre de l’Économie, des Finances et de la Souveraineté industrielle, énergétique et numérique, chargée de l’Intelligence artificielle et du Numérique
- Marina Ferrari : Ministre des Sports, de la Jeunesse et de la Vie associative
- Eléonore Caroit : Ministre déléguée auprès du ministre de l’Europe et des Affaires étrangères, chargée de la Francophonie, des Partenariats internationaux et des Français de l’étranger
- Mathieu Lefèvre : Ministre délégué auprès de la ministre de la Transition écologique, de la Biodiversité et des Négociations internationales sur le climat et la nature, chargé de la Transition écologique
- David Amiel : Ministre de l’Action et des Comptes publics
- Benjamin Haddad : Ministre délégué auprès du ministre de l’Europe et des Affaires étrangères, chargé de l’Europe
- Maud Bregeon : Ministre déléguée, porte-parole du Gouvernement auprès du Premier ministre et ministre déléguée, chargée de l'Énergie auprès du ministre de l'Économie, des Finances et de la Souveraineté industrielle, énergétique et numérique
- Vincent Jeanbrun : Ministre de la Ville et du Logement
- Sébastien Martin : Ministre délégué auprès du ministre de l’Économie, des Finances et de la Souveraineté industrielle, énergétique et numérique, chargé de l'Industrie



Il est possible de filtrer cette liste de ministres en choisissant circonscription.numCirco = Null

In [14]:
df_tr[df_tr["circonscription.numCirco"].notnull()]

,uid,nom,prenom,civ,chambre,actif,urlImage,interets,placeHemicycle,compteTwitter,...,circonscription.uid,circonscription.numCirco,circonscription.numDepartement,circonscription.libelle,commissionPermanente.uid,commissionPermanente.libelle,commissionPermanente.libelleAbrege,groupeParlementaire,circonscription,commissionPermanente
0,PA1008,David,Alain,M.,AN,True,https://tricoteuses-assets.s3.fr-par.scw.cloud...,"[{'categorie': 'ruralite', 'label': 'Ruralité ...",410,NaN,...,PO839429,4.0,33.0,4ème circonscription de la Gironde,PO59047,Commission des affaires étrangères,Affaires étrangères,NaN,NaN,NaN
2,PA1567,Guedj,Jérôme,M.,AN,True,https://tricoteuses-assets.s3.fr-par.scw.cloud...,"[{'categorie': 'social', 'label': 'Protection ...",417,@JeromeGuedj,...,PO839989,6.0,91.0,6ème circonscription de l'Essonne,PO420120,Commission des affaires sociales,Affaires sociales,NaN,NaN,NaN
3,PA1592,Habib,David,M.,AN,True,https://tricoteuses-assets.s3.fr-par.scw.cloud...,"[{'categorie': 'ruralite', 'label': 'Ruralité ...",455,@DavidDhabib,...,PO839701,3.0,64.0,3ème circonscription des Pyrénées-Atlantiques,PO59046,Commission de la défense nationale et des forc...,Défense,NaN,NaN,NaN
4,PA1630,Herbillon,Michel,M.,AN,True,https://tricoteuses-assets.s3.fr-par.scw.cloud...,"[{'categorie': 'ruralite', 'label': 'Ruralité ...",175,@Michelherbillon,...,PO838957,8.0,94.0,8ème circonscription du Val-de-Marne,PO59047,Commission des affaires étrangères,Affaires étrangères,NaN,NaN,NaN
5,PA1654,Hollande,François,M.,AN,True,https://tricoteuses-assets.s3.fr-par.scw.cloud...,"[{'categorie': 'social', 'label': 'Protection ...",424,NaN,...,PO839227,1.0,19.0,1ère circonscription de la Corrèze,PO59047,Commission des affaires étrangères,Affaires étrangères,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590,PA857021,Latrèche,Alim,M.,AN,True,https://tricoteuses-assets.s3.fr-par.scw.cloud...,"[{'categorie': 'ruralite', 'label': 'Ruralité ...",397,NaN,...,PO839537,1.0,38.0,1ère circonscription de l'Isère,PO420120,Commission des affaires sociales,Affaires sociales,NaN,NaN,NaN
591,PA870009,Duparay,Lionel,M.,AN,True,https://tricoteuses-assets.s3.fr-par.scw.cloud...,"[{'categorie': 'sport', 'label': 'Sport', 'sco...",209,NaN,...,PO839795,5.0,71.0,5ème circonscription de la Saône-et-Loire,PO419604,Commission des affaires culturelles et de l'éd...,Affaires culturelles et éducation,NaN,NaN,NaN
593,PA878242,Valentin,Antoine,M.,AN,True,https://tricoteuses-assets.s3.fr-par.scw.cloud...,"[{'categorie': 'ruralite', 'label': 'Ruralité ...",102,NaN,...,PO839755,3.0,74.0,3ème circonscription de la Haute-Savoie,PO59046,Commission de la défense nationale et des forc...,Défense,NaN,NaN,NaN
594,PA879389,Lubet,Marie-Philippe,Mme,AN,True,https://tricoteuses-assets.s3.fr-par.scw.cloud...,"[{'categorie': 'ruralite', 'label': 'Ruralité ...",304,@Lubet_MP,...,PO839367,1.0,45.0,1ère circonscription du Loiret,PO419865,Commission du développement durable et de l'am...,Développement durable,NaN,NaN,NaN


In [15]:
df_dn[df_dn["prenom"]=="Catherine"]

,__id,id,legislature,civ,nom,prenom,villeNaissance,naissance,age,groupe,...,twitter,facebook,website,nombreMandats,experienceDepute,scoreParticipation,scoreParticipationSpecialite,scoreLoyaute,scoreMajorite,dateMaj
402,403,PA795954,17,Mme,Ibled,Catherine,Neuilly-Sur-Seine,1975-05-25,51,Ensemble pour la République,...,NaN,NaN,NaN,1,1 an,0.27,0.31,0.980,0,2026-10-28
458,459,PA840947,17,Mme,Hervieu,Catherine,Meknès,1958-05-09,68,Écologiste et Social,...,NaN,catherine.hervieu.9,catherinehervieu.fr,1,2 ans,0.31,0.85,0.962,0,2026-10-28
466,467,PA841059,17,Mme,Dellong Meng,Catherine,Avignon,1960-05-04,66,Rassemblement National,...,NaN,NaN,NaN,1,1 an,0.40,0.80,0.984,0,2026-10-28
542,543,PA841981,17,Mme,Rimbert,Catherine,Avignon,1968-09-13,58,Rassemblement National,...,@cathRimbert84,Catherine Rimbert,NaN,1,2 ans,0.48,0.90,0.994,0,2026-10-28


In [16]:
df_tr[df_tr["prenom"]=="Catherine"]

,uid,nom,prenom,civ,chambre,actif,urlImage,interets,placeHemicycle,compteTwitter,...,circonscription.uid,circonscription.numCirco,circonscription.numDepartement,circonscription.libelle,commissionPermanente.uid,commissionPermanente.libelle,commissionPermanente.libelleAbrege,groupeParlementaire,circonscription,commissionPermanente
17,PA267797,Vautrin,Catherine,Mme,AN,True,https://tricoteuses-assets.s3.fr-par.scw.cloud...,"[{'categorie': 'ruralite', 'label': 'Ruralité ...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
420,PA795954,Ibled,Catherine,Mme,AN,True,https://tricoteuses-assets.s3.fr-par.scw.cloud...,"[{'categorie': 'famille', 'label': 'Famille', ...",280,NaN,...,PO839787,13.0,75.0,13ème circonscription de Paris,PO419604,Commission des affaires culturelles et de l'éd...,Affaires culturelles et éducation,NaN,NaN,NaN
478,PA840947,Hervieu,Catherine,Mme,AN,True,https://tricoteuses-assets.s3.fr-par.scw.cloud...,"[{'categorie': 'immigration', 'label': 'Immigr...",531,NaN,...,PO839211,2.0,21.0,2ème circonscription de la Côte-d'Or,PO59046,Commission de la défense nationale et des forc...,Défense,NaN,NaN,NaN
486,PA841059,Dellong Meng,Catherine,Mme,AN,True,https://tricoteuses-assets.s3.fr-par.scw.cloud...,"[{'categorie': 'ruralite', 'label': 'Ruralité ...",080,NaN,...,PO839365,3.0,30.0,3ème circonscription du Gard,PO59048,"Commission des finances, de l'économie général...",Finances,NaN,NaN,NaN
562,PA841981,Rimbert,Catherine,Mme,AN,True,https://tricoteuses-assets.s3.fr-par.scw.cloud...,"[{'categorie': 'ruralite', 'label': 'Ruralité ...",040,@cathRimbert84,...,PO839149,5.0,84.0,5ème circonscription de Vaucluse,PO59046,Commission de la défense nationale et des forc...,Défense,NaN,NaN,NaN


# Potentiels Cas particuliers à gérer : 
- Députés dont le mandat est clos : https://www2.assemblee-nationale.fr/deputes/liste/clos

### Loïc Kervran (Cher) pour 'Election au Sénat' le 30/09/2026

In [17]:
#source : datan
df_dn[df_dn["nom"]=="Kervran"]

,__id,id,legislature,civ,nom,prenom,villeNaissance,naissance,age,groupe,...,twitter,facebook,website,nombreMandats,experienceDepute,scoreParticipation,scoreParticipationSpecialite,scoreLoyaute,scoreMajorite,dateMaj


In [18]:
#source : tricoteuses
df_tr[df_tr["nom"]=="Kervran"]

,uid,nom,prenom,civ,chambre,actif,urlImage,interets,placeHemicycle,compteTwitter,...,circonscription.uid,circonscription.numCirco,circonscription.numDepartement,circonscription.libelle,commissionPermanente.uid,commissionPermanente.libelle,commissionPermanente.libelleAbrege,groupeParlementaire,circonscription,commissionPermanente


### Hervé Berville (Côté d'Armor) pour 'Prolongation de mission temporaire' le 04/08/2026

In [19]:
#source : datan
df_dn[df_dn["nom"]=="Berville"]

,__id,id,legislature,civ,nom,prenom,villeNaissance,naissance,age,groupe,...,twitter,facebook,website,nombreMandats,experienceDepute,scoreParticipation,scoreParticipationSpecialite,scoreLoyaute,scoreMajorite,dateMaj


In [20]:
#source : tricoteuses
df_tr[df_tr["nom"]=="Berville"]

,uid,nom,prenom,civ,chambre,actif,urlImage,interets,placeHemicycle,compteTwitter,...,circonscription.uid,circonscription.numCirco,circonscription.numDepartement,circonscription.libelle,commissionPermanente.uid,commissionPermanente.libelle,commissionPermanente.libelleAbrege,groupeParlementaire,circonscription,commissionPermanente


### Harold Huwart (Eure-et-Loir) pour 'Démission pour cause d’incompatibilité prévue aux articles LO 137, LO 137-1, LO 141 ou LO 141-1 du code électoral' (Elu Maire) le 25/07/2026

In [21]:
#source : datan
df_dn[df_dn["nom"]=="Huwart"]

,__id,id,legislature,civ,nom,prenom,villeNaissance,naissance,age,groupe,...,twitter,facebook,website,nombreMandats,experienceDepute,scoreParticipation,scoreParticipationSpecialite,scoreLoyaute,scoreMajorite,dateMaj


In [22]:
#source : tricoteuses
df_tr[df_tr["nom"]=="Huwart"]

,uid,nom,prenom,civ,chambre,actif,urlImage,interets,placeHemicycle,compteTwitter,...,circonscription.uid,circonscription.numCirco,circonscription.numDepartement,circonscription.libelle,commissionPermanente.uid,commissionPermanente.libelle,commissionPermanente.libelleAbrege,groupeParlementaire,circonscription,commissionPermanente


### Jean-Pierre Taite (Loire) pour 'Démission' le 08/07/2026

In [23]:
#source : datan
df_dn[df_dn["nom"]=="Taite"]

,__id,id,legislature,civ,nom,prenom,villeNaissance,naissance,age,groupe,...,twitter,facebook,website,nombreMandats,experienceDepute,scoreParticipation,scoreParticipationSpecialite,scoreLoyaute,scoreMajorite,dateMaj


In [24]:
#source : tricoteuses
df_tr[df_tr["nom"]=="Taite"]

,uid,nom,prenom,civ,chambre,actif,urlImage,interets,placeHemicycle,compteTwitter,...,circonscription.uid,circonscription.numCirco,circonscription.numDepartement,circonscription.libelle,commissionPermanente.uid,commissionPermanente.libelle,commissionPermanente.libelleAbrege,groupeParlementaire,circonscription,commissionPermanente


### Béatrice Bellamy (Vendée) pour 'Décès' le 24/05/2026

In [25]:
#source : datan
df_dn[df_dn["nom"]=="Bellamy"]

,__id,id,legislature,civ,nom,prenom,villeNaissance,naissance,age,groupe,...,twitter,facebook,website,nombreMandats,experienceDepute,scoreParticipation,scoreParticipationSpecialite,scoreLoyaute,scoreMajorite,dateMaj


In [26]:
#source : tricoteuses
df_tr[df_tr["nom"]=="Bellamy"]

,uid,nom,prenom,civ,chambre,actif,urlImage,interets,placeHemicycle,compteTwitter,...,circonscription.uid,circonscription.numCirco,circonscription.numDepartement,circonscription.libelle,commissionPermanente.uid,commissionPermanente.libelle,commissionPermanente.libelleAbrege,groupeParlementaire,circonscription,commissionPermanente


### Jean Moulière (Nord) pour 'Reprise de l'exercice du mandat d'un ancien membre du Gouvernement' le 26/03/2026

In [27]:
#source : datan
df_dn[df_dn["nom"]=="Moulière"]

,__id,id,legislature,civ,nom,prenom,villeNaissance,naissance,age,groupe,...,twitter,facebook,website,nombreMandats,experienceDepute,scoreParticipation,scoreParticipationSpecialite,scoreLoyaute,scoreMajorite,dateMaj


In [28]:
#source : tricoteuses
df_tr[df_tr["nom"]=="Moulière"]

,uid,nom,prenom,civ,chambre,actif,urlImage,interets,placeHemicycle,compteTwitter,...,circonscription.uid,circonscription.numCirco,circonscription.numDepartement,circonscription.libelle,commissionPermanente.uid,commissionPermanente.libelle,commissionPermanente.libelleAbrege,groupeParlementaire,circonscription,commissionPermanente


### Jean-Didier Berger (Haut-de-Seine) pour 'Nomination comme membre du Gouvernement' le 26/02/2026

In [29]:
#source : datan
df_dn[df_dn["nom"]=="Berger"]

,__id,id,legislature,civ,nom,prenom,villeNaissance,naissance,age,groupe,...,twitter,facebook,website,nombreMandats,experienceDepute,scoreParticipation,scoreParticipationSpecialite,scoreLoyaute,scoreMajorite,dateMaj


In [30]:
#source : tricoteuses
df_tr[df_tr["nom"]=="Berger"]

,uid,nom,prenom,civ,chambre,actif,urlImage,interets,placeHemicycle,compteTwitter,...,circonscription.uid,circonscription.numCirco,circonscription.numDepartement,circonscription.libelle,commissionPermanente.uid,commissionPermanente.libelle,commissionPermanente.libelleAbrege,groupeParlementaire,circonscription,commissionPermanente
41,PA408578,Berger,Jean-Didier,M.,AN,True,https://tricoteuses-assets.s3.fr-par.scw.cloud...,"[{'categorie': 'logement', 'label': 'Logement'...",NaN,@JD_Berger,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Jean Moulière (Nord) pour 'Reprise de l'exercice du mandat d'un ancien membre du Gouvernement' le 26 mars 2026

In [31]:
#source : datan
df_dn[df_dn["nom"]=="Moulière"]

,__id,id,legislature,civ,nom,prenom,villeNaissance,naissance,age,groupe,...,twitter,facebook,website,nombreMandats,experienceDepute,scoreParticipation,scoreParticipationSpecialite,scoreLoyaute,scoreMajorite,dateMaj


In [32]:
#source : tricoteuses
df_tr[df_tr["nom"]=="Moulière"]

,uid,nom,prenom,civ,chambre,actif,urlImage,interets,placeHemicycle,compteTwitter,...,circonscription.uid,circonscription.numCirco,circonscription.numDepartement,circonscription.libelle,commissionPermanente.uid,commissionPermanente.libelle,commissionPermanente.libelleAbrege,groupeParlementaire,circonscription,commissionPermanente


## Conclusion : les cas particuliers à l'exception des députés passés ministres sont géré de la même façon entre les deux sources

# Import data from Assemblée nationale 

### count deputy in source

In [33]:
import io #module to call BytesIO, it creates a in-memorys binary stream allowing to use raw in-memory instead of a file object
import zipfile

URL_ZIP = "https://data.assemblee-nationale.fr/static/openData/repository/17/amo/deputes_actifs_mandats_actifs_organes/AMO10_deputes_actifs_mandats_actifs_organes.json.zip"

reponse = requests.get(URL_ZIP, timeout=60)
fichier_zip = zipfile.ZipFile(io.BytesIO(reponse.content))

# Garder seulement les fichiers des députés
fichiers_deputes = [nom for nom in fichier_zip.namelist() if "/acteur/" in nom]

print(len(fichiers_deputes))
print(fichiers_deputes)

574
['json/acteur/PA841605.json', 'json/acteur/PA793872.json', 'json/acteur/PA841553.json', 'json/acteur/PA607193.json', 'json/acteur/PA267042.json', 'json/acteur/PA795156.json', 'json/acteur/PA795176.json', 'json/acteur/PA842085.json', 'json/acteur/PA720138.json', 'json/acteur/PA720614.json', 'json/acteur/PA841395.json', 'json/acteur/PA368.json', 'json/acteur/PA720560.json', 'json/acteur/PA795864.json', 'json/acteur/PA267780.json', 'json/acteur/PA266808.json', 'json/acteur/PA841191.json', 'json/acteur/PA718674.json', 'json/acteur/PA840947.json', 'json/acteur/PA2940.json', 'json/acteur/PA794038.json', 'json/acteur/PA795664.json', 'json/acteur/PA720146.json', 'json/acteur/PA796062.json', 'json/acteur/PA793382.json', 'json/acteur/PA607619.json', 'json/acteur/PA794778.json', 'json/acteur/PA793398.json', 'json/acteur/PA720092.json', 'json/acteur/PA795974.json', 'json/acteur/PA588884.json', 'json/acteur/PA793924.json', 'json/acteur/PA793616.json', 'json/acteur/PA794802.json', 'json/acteur/P

In [34]:
import json
data_PA841605 = json.load(fichier_zip.open("json/acteur/PA841605.json"))

# write the json file to a local file
with open("PA841605.json", "w") as f:
    json.dump(data_PA841605, f, ensure_ascii=False, indent=4)